In [3]:
'''

import matplotlib.pyplot as plt
from types import SimpleNamespace

from sklearn.manifold import TSNE'''

'import os\n\nimport matplotlib.pyplot as plt\nfrom types import SimpleNamespace\n\nfrom sklearn.manifold import TSNE'

In [23]:
import os
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [24]:
# Variables:
min_triggers = 20
freq_dist = 0.2

# Parameters:
clust_dim = 1
temp_dist = 0.1
deltat = 2.0
filterr = False

clust, run = 'teste', 'O3a'
sufixo = '_filtered_' if filterr else '_'

In [25]:
base = os.getcwd()
repository = base + '/zrepository'
temp = repository + '/' + clust + '/' + run + '/' + str(clust_dim) + 'D_dailyClustering_c' + str(min_triggers)

daily_clustering = (temp + '_f' + str(freq_dist) if clust_dim == 2 else temp) + '_t' + str(temp_dist)

In [26]:
daily_clustering

'/home/esanches/Projetos/tSNE/Virgo/zrepository/teste/O3a/1D_dailyClustering_c20_t0.1'

## Functions

In [3]:
def glitch_to_glitchgram(unclustered, clustered, deltat, 
                         fmin=3, fmax=8192, tbins=41, fbins=30):
    """
    This function create one glitchgram from unclustered Omicron data, for each glitch.

    Parameters:
        df_triggers (dataFrame): A dataFrame of GPStimes of all triggers inside of some window.
        tmin, tmax (numbers): limits of GPStimes that contain all triggers of some glitch.
        fmin, fmax (numbers): frequency range for the glitchgram.
        n_time_bins (int): number of time bins.
        n_freq_bins (int): number of frequency bins.
        norm (boolean): True if you need normalize SNR values.

    Returns:
        array for some glitch: each entry is a SNR value from flattened (binsf × binst) matrix.
    """

    times_all = unclustered['time'].values
    freqs_all = unclustered['frequency'].values
    snrs_all  = unclustered['snr'].values

    vecs = []
    labs = []
    GPSs = []

    time_edges = np.linspace(-deltat/2, deltat/2, tbins+1)
    freq_edges = np.logspace(np.log10(fmin), np.log10(fmax), fbins+1)
    
    for i, row in enumerate(clustered.itertuples(index=False)):

        #t_center = getattr(row, 'GPStime')
        t_center = row.time
    
        tmin = t_center - deltat/2
        tmax = t_center + deltat/2
    
        i0 = np.searchsorted(times_all, tmin, side="left")
        i1 = np.searchsorted(times_all, tmax, side="right")
    
        times = times_all[i0:i1] - t_center
        freqs = freqs_all[i0:i1]
        snrs  = snrs_all[i0:i1]

        if len(times) < 10: # Equivalente ao tcut
            continue
    
        df_triggers = pd.DataFrame({'time': times,'frequency': freqs,'snr': snrs})
        
        t_idx = np.digitize(df_triggers['time'].values, time_edges) - 1
        f_idx = np.digitize(df_triggers['frequency'].values, freq_edges) - 1
            
        # initial empty matrix
        A = np.zeros((fbins, tbins), dtype=float)
    
        for ti, fi, si in zip(t_idx, f_idx, snrs):
            if 0 <= ti < tbins and 0 <= fi < fbins:
                if si > A[fi, ti]:
                    A[fi, ti] = si
                    
        '''# normalização do SNR 
        if len(snrs) > 0:
            snr_min = snrs.min()
            snr_max = snrs.max()
            if snr_max > snr_min:
                norm_snrs = (snrs - snr_min) / (snr_max - snr_min)
            else:
                norm_snrs = np.zeros_like(snrs)
        else:
            norm_snrs = snrs'''
        
        vecs.append(A.flatten(order="C"))
        labs.append(row.label)
        GPSs.append(row.time)

    vecs_final = np.array(vecs).astype('float32')
    labs_final = np.array(labs)
    GPSs_final = np.array(GPSs).astype('float32')
    
    return vecs_final, labs_final, GPSs_final

## Applying filters

* To see the arquive lenght in CMD use: **wc -l arquivename.csv**
* To see the arquive columns in CMD use: **head -n 1 arquivename.csv**

Triggers

In [5]:
# trigcols = ['time', 'frequency', 'snr']

'''dtypes = {
    'time': 'float64',
    'frequency': 'float32',
    'snr': 'float32'
}'''

chunk_size = 10_000_000
chunks, count = [], 0

for chunk in pd.read_csv(base + '/triggers_total.csv', dtype=dtypes, chunksize=chunk_size):

    if count == 2:
        break
    print(f'Reading chunk: {count}')
    chunks.append(chunk)
    count += 1

trigg = pd.concat(chunks, ignore_index=True)
trigg = trigg.sort_values('time').reset_index(drop=True)

print(f"Total triggers successfully loaded: {len(trigg)}")

Reading chunk: 0
Reading chunk: 1
Total triggers successfully loaded: 20000000


In [7]:
trigg

,time,frequency,snr
0,1.256655e+09,957.826904,5.240905
1,1.256655e+09,957.826904,5.026537
2,1.256655e+09,349.472961,5.334844
3,1.256655e+09,349.472961,5.754359
4,1.256655e+09,313.632324,5.250525
...,...,...,...
19999995,1.261000e+09,30.399595,5.606653
19999996,1.261000e+09,28.713770,6.651165
19999997,1.261000e+09,28.758080,5.875793
19999998,1.261000e+09,29.177448,6.025537


In [5]:
unclustered = trigg.copy()
unclustered.tail()

,time,frequency,snr
359385796,1.269672e+09,583.020752,38.427120
359385797,1.269672e+09,830.530640,11.973112
359385798,1.269672e+09,474.735565,55.750835
359385799,1.269672e+09,1079.892944,15.370767
359385800,1.269672e+09,879.323059,19.674751


In [8]:
'''tperc, trigg_size = 0.7, 359385802
trigcols = ['time', 'frequency', 'snr']
ntrig = int(trigg_size*tperc)

""" Performs a random and filtered read of a large CSV file to save memory """

np.random.seed(42)
skip_logic = lambda i: i > 0 and np.random.random() > tperc
trigg = pd.read_csv(f'{base_data}triggers_total.csv', skiprows=skip_logic, usecols=trigcols)
trigg = trigg.sort_values('time').reset_index(drop=True)
unclustered = trigg.copy()'''

## Gravity Spy clustering

In [9]:
gspy = pd.read_csv(base + '/' + clust + run + '.csv')
gspy = gspy[gspy['confidence'] >= 0.9]
gspy = gspy.drop(gspy[gspy['label'].isin(['None_of_the_Above', 'No_Glitch'])].index).reset_index(drop=True)
gspy = gspy.rename(columns={'GPStime':'time'})

top10_labels = gspy['label'].value_counts().nlargest(10).index
gspy_top10 = gspy[gspy['label'].isin(top10_labels)]

if filterr == True:
    limited_glit = gspy_top10.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42)).reset_index(drop=True)
    limited_glit = limited_glit.sort_values(by=['time']).reset_index(drop=True)
            
    clustered = limited_glit.copy()
else:
    clustered = gspy.copy()

clustered

,time,peakFreq,snr,amplitude,centralFreq,duration,bandwidth,chisq,chisqDof,confidence,id,ifo,label,imgUrl,Q-value
0,1.239157e+09,62.976,19989.938,2.580000e-19,3983.241,13.524,7934.482910,0.0,0.0,1.0,R7q36rPK5J,V1,Extremely_Loud,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,11.314
1,1.239923e+09,77.754,10047.964,1.070000e-19,3983.241,10.250,7934.482910,0.0,0.0,1.0,dY3C2ZM8m0,V1,Extremely_Loud,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,11.314
2,1.239912e+09,77.754,7198.535,7.480000e-20,3983.241,15.500,7934.482910,0.0,0.0,1.0,x9XOy9akFC,V1,Extremely_Loud,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,11.314
3,1.243488e+09,42.513,6134.880,1.690000e-19,3983.241,44.250,7934.482910,0.0,0.0,1.0,VVTDP72IC0,V1,Extremely_Loud,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,45.255
4,1.251572e+09,77.754,6099.467,6.790000e-20,3983.241,11.750,7934.482910,0.0,0.0,1.0,2FPtnepKVC,V1,Extremely_Loud,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,11.314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41426,1.238478e+09,24.329,9.116,1.540000e-21,1267.535,2.000,2503.070313,0.0,0.0,0.9,BWL5uUnNyu,V1,Scattered_Light,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,22.627
41427,1.253434e+09,19.292,8.315,1.750000e-21,1161.367,1.500,2290.733643,0.0,0.0,0.9,Djty5Vsg79,V1,Low_Frequency_Lines,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,45.255
41428,1.244794e+09,422.792,9.344,1.590000e-22,4014.908,0.347,7281.335449,0.0,0.0,0.9,bfA6S3f6iy,V1,Whistle,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,45.255
41429,1.239700e+09,27.033,8.944,5.990000e-22,3985.119,1.292,7930.728027,0.0,0.0,0.9,uoj0FaSv2R,V1,Scattered_Light,https://ldas-jobs.ligo.caltech.edu/~gravityspy...,45.255


In [12]:
'''gperc, glit_size = 1.0, 187770
glitcols = ['GPStime', 'centralFreq', 'snr', 'label', 'confidence']
nglit = int(glit_size*gperc)

""" Apply filters to the glitches """

glit = pd.read_csv(f'{base_data}{arq_name}.csv', usecols=glitcols)

glit = glit[glit['confidence'] >= 0.9]
glit = glit.drop(glit[glit['label'].isin(['None_of_the_Above', 'No_Glitch'])].index)

top10 = glit['label'].value_counts().head(10).index
glit = glit[glit['label'].isin(top10)].copy()

glit = glit.sort_values(by=[glitcols[0]]).reset_index(drop=True)

if filterr == True:
    limited_glit = glit.groupby('label', group_keys=False).apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))
    limited_glit = limited_glit.sort_values(by=['GPStime']).reset_index(drop=True)
            
    clustered = limited_glit.copy()
else:
    clustered = glit.copy()

clustered.to_csv(f'{repository}clustered_{arq_name}.csv')'''

## Detchar clustering

* COLUMNS: Unnamed: 0, cluster, tstart, tend, fmin, fmax, time, frequency, q, snr, phase

In [31]:
glitcols = ['time', 'frequency', 'snr']
glit = pd.read_csv(base + '/' + clust + run + '_' + cut + '.csv', usecols=glitcols)

FileNotFoundError: [Errno 2] No such file or directory: '/home/esanches/Projetos/tSNE/Virgo/testeO3a_cut20.csv'

In [24]:
if filterr == True:
    limited_glit = glit[(glit['snr'] >= 7.5)]
    limited_glit = limited_glit.sample(n=min(len(limited_glit), 10000), random_state=42)
    
    limited_glit = limited_glit.sort_values(by=['time']).reset_index(drop=True)
            
    clustered = limited_glit.copy()
else:
    limited_glit = glit[(glit['snr'] >= 7.5)]
    clustered = limited_glit.copy()

In [25]:
repository

'/home/esanches/Projetos/tSNE/Virgo/zrepository/gdchar/O3a/'

In [26]:
clustered.to_csv(repository + 'clustered' + sufixo + cut + '.csv', index=False)
clustered

,time,frequency,snr
1,1.238166e+09,138.375243,11.157326
3,1.238167e+09,19.453599,8.515493
4,1.238167e+09,21.824261,8.541214
7,1.238168e+09,19.494597,7.698744
9,1.238168e+09,336.991363,10.627294
...,...,...,...
131194,1.253975e+09,32.842241,12.071470
131199,1.253977e+09,1079.892919,89.686901
131200,1.253977e+09,20.025103,8.843882
131201,1.253977e+09,20.317121,8.158477


In [90]:
clustered['snr'].max()

np.float64(1552.1200362489617)

## vectorization

In [8]:
vecs, labs, GPSs = glitch_to_glitchgram(unclustered, clustered, dt, fmin=3, fmax=8192, tbins=41, fbins=30)

In [9]:
np.save(repository + 'vecs_' + clust + run + '_dt' + str(dt), vecs)
np.save(repository + 'labs_' + clust + run + '_dt' + str(dt), labs)
np.save(repository + 'GPSs_' + clust + run + '_dt' + str(dt), GPSs)